In [1]:
import os

# Fix: Keras/TensorFlow can crash with protobuf C++ bindings mismatch in some Kaggle images.
# Forcing the pure-Python protobuf implementation avoids:
#   AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import glob
import random
from enum import Enum

import numpy as np
import pandas as pd

import cv2
import pydicom

import tensorflow as tf
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.metrics import AUC
from tensorflow.keras.utils import Sequence
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.layers import (
    Input,
    Conv3D,
    BatchNormalization,
    MaxPool3D,
    Dense,
    Dropout,
    concatenate,
    GlobalAveragePooling3D,
    ReLU,
)




2026-05-13 03:58:42.351538: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-13 03:58:42.362304: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778644722.374028      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778644722.377429      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-13 03:58:42.390639: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
class InternalDebug:
    def __init__(self, debug_mode=False, debug_prefix=None):
        self.__debug_mode = debug_mode
        self.__debug_prefix = debug_prefix

    @property
    def __prefix(self):
        if self.__debug_prefix is not None:
            return self.__debug_prefix
        else:
            return ""

    def log(self, *args):
        if self.__debug_mode:
            if self.__debug_prefix is not None:
                print(self.__debug_prefix, *args)
            else:
                print(*args)

    def separator(self, character="=", length=50):
        separator = character * length
        self.log(separator)

    def info(self, *args):
        if self.__debug_mode:
            prefix = self.__prefix + "[INFO]"
            self.log(prefix, *args)

    def warning(self, *args):
        if self.__debug_mode:
            prefix = self.__prefix + "[WARNING]"
            self.log(prefix, *args)

    def error(self, *args):
        if self.__debug_mode:
            prefix = self.__prefix + "[ERROR]"
            self.log(prefix, *args)

    def set_debug_mode(self, debug_mode):
        self.__debug_mode = debug_mode




In [3]:
IS_PYDICOM_IMPORTED = True
if IS_PYDICOM_IMPORTED:
    try:
        from concurrent.futures import ThreadPoolExecutor, as_completed
    except ImportError:
        ThreadPoolExecutor = None
        as_completed = None

    class ImageFormat(Enum):
        WHDC = "W-H-D-C"  # (W,H,D,C)
        DWHC = "D-W-H-C"  # (D,W,H,C)

        def swap_dimensions(image, image_format):
            if image_format == ImageFormat.DWHC:
                image = np.transpose(image, (2, 1, 0, 3))
            elif image_format == ImageFormat.WHDC:
                image = np.transpose(image, (2, 1, 0, 3))
            return image

    class DICOMLoader:
        def __init__(
            self,
            df,
            input_path,
            scan_categories,
            num_imgs=None,
            size=(224, 224),
            scale=1.0,
            rotate_angle=0,
            enable_center_focus=False,
            id_column_name="ID",
            label_column_name="Label",
            image_format=ImageFormat.WHDC,
            max_threads=8,
            image_file_sorter=lambda x: int(x[:-4].split("-")[-1]),
            debug_mode=False,
            cache_dir=None,
            memory_cache_size=200_000,
        ):
            if num_imgs is not None and num_imgs % 2 != 0 and enable_center_focus:
                raise ValueError("num_imgs must be divisible by 2 for central image")

            if not (0 <= rotate_angle <= 360):
                raise ValueError("Rotation value must be between 0 and 360")

            for col in [id_column_name, label_column_name]:
                if col not in df.columns:
                    raise ValueError(f"Columns {col} must be in dataset")

            self.__ids = df[id_column_name].to_numpy()
            self.__labels = df[label_column_name].to_numpy()

            self.__num_imgs = num_imgs
            self.__id_column_name = id_column_name
            self.__label_column_name = label_column_name
            self.__input_path = input_path
            self.__scan_categories = scan_categories
            self.__max_threads = max_threads
            self.__size = size
            self.__scale = scale
            self.__rotate_angle = rotate_angle
            self.__image_format = image_format
            self.__image_file_sorter = image_file_sorter
            self.__enable_center_focus = enable_center_focus
            self.__debug = InternalDebug(debug_mode=debug_mode)

            self.__cache_dir = cache_dir
            if self.__cache_dir is not None:
                os.makedirs(self.__cache_dir, exist_ok=True)
            self.__mem_cache = {}
            self.__mem_cache_fifo = []
            self.__mem_cache_size = int(memory_cache_size)

        @property
        def scan_categories(self):
            return self.__scan_categories

        @property
        def num_imgs(self):
            return self.__num_imgs

        @property
        def image_format(self):
            return self.__image_format

        @property
        def df(self):
            return pd.DataFrame(
                {
                    self.__id_column_name: self.__ids,
                    self.__label_column_name: self.__labels,
                }
            )

        @property
        def len(self):
            return len(self.__ids)

        def get_id(self, row):
            return self.__ids[row]

        def gel_label(self, row):
            return self.__labels[row]

        def format(self, images, type):
            if type == "normalize" and self.image_format == ImageFormat.WHDC:
                return ImageFormat.swap_dimensions(images, ImageFormat.DWHC)
            elif type == "default" and self.image_format == ImageFormat.WHDC:
                return ImageFormat.swap_dimensions(images, ImageFormat.WHDC)
            else:
                return images

        def load_scan(self, row, scan_category, show_progress=True):
            patient_id = str(self.__ids[row]).zfill(5)
            scans_path = os.path.join(self.__input_path, patient_id, scan_category)

            if not os.path.exists(scans_path):
                raise FileNotFoundError(f"The folder {scans_path} doesn't exist.")

            image_files = sorted(
                glob.glob(os.path.join(scans_path, "*")), key=self.__image_file_sorter
            )
            if not image_files:
                raise ValueError(f"No image files found in {scans_path}.")

            image_files = self._select_subset_image_files(image_files)

            loaded_images = []

            if ThreadPoolExecutor is None:
                for p in image_files:
                    loaded_images.append(self._load_dicom_image_cached(p))
            else:
                with ThreadPoolExecutor(self.__max_threads) as executor:
                    for image_data in executor.map(
                        self._load_dicom_image_cached, image_files
                    ):
                        if image_data is not None:
                            loaded_images.append(image_data)

            if not loaded_images:
                raise ValueError("No images were loaded. Cannot proceed.")

            if self.__num_imgs is not None:
                while len(loaded_images) < self.__num_imgs:
                    loaded_images.append(np.zeros_like(loaded_images[0]))

            loaded_images = np.array(loaded_images)
            return self.format(loaded_images, "default")

        def load_all_scans(self, row, show_progress=True):
            all_images = {}

            if ThreadPoolExecutor is None or as_completed is None:
                for scan_category in self.__scan_categories:
                    all_images[scan_category] = self.load_scan(
                        row, scan_category, show_progress=False
                    )
                return {key: all_images.get(key, []) for key in self.__scan_categories}

            with ThreadPoolExecutor(self.__max_threads) as executor:
                future_to_scan_category = {
                    executor.submit(
                        self.load_scan, row, scan_category, False
                    ): scan_category
                    for scan_category in self.__scan_categories
                }

                for future in as_completed(future_to_scan_category):
                    scan_category = future_to_scan_category[future]
                    image_data = future.result()
                    if image_data is not None:
                        all_images[scan_category] = image_data

            return {key: all_images.get(key, []) for key in self.__scan_categories}

        def _load_dicom_image_cached(self, dicom_path):
            key = (
                dicom_path,
                self.__size,
                float(self.__scale),
                float(self.__rotate_angle),
            )

            v = self.__mem_cache.get(key)
            if v is not None:
                return v

            cache_file = None
            if self.__cache_dir is not None:
                cache_file = os.path.join(self.__cache_dir, f"{abs(hash(key))}.npy")
                if os.path.exists(cache_file):
                    arr = np.load(cache_file, allow_pickle=False)
                    self._mem_cache_put(key, arr)
                    return arr

            arr = self._load_dicom_image(dicom_path)

            if cache_file is not None:
                tmp = cache_file + ".tmp"
                np.save(tmp, arr, allow_pickle=False)
                os.replace(tmp, cache_file)

            self._mem_cache_put(key, arr)
            return arr

        def _mem_cache_put(self, key, arr):
            self.__mem_cache[key] = arr
            self.__mem_cache_fifo.append(key)
            if len(self.__mem_cache_fifo) > self.__mem_cache_size:
                old = self.__mem_cache_fifo.pop(0)
                self.__mem_cache.pop(old, None)

        def _load_dicom_image(self, dicom_path):
            if not os.path.exists(dicom_path):
                raise FileNotFoundError(f"File {dicom_path} does not exist.")

            dicom_file = pydicom.dcmread(dicom_path)
            image = dicom_file.pixel_array

            image = self._rotate_img(image)
            image = self._normalization_img(image)
            image = self._crop_img(image)
            image = self._resize_img(image)
            image = np.expand_dims(image, axis=-1)
            return image

        def _resize_img(self, image):
            w, h = self.__size
            return cv2.resize(image, (w, h), interpolation=cv2.INTER_AREA)

        def _crop_img(self, image):
            if self.__scale <= 0:
                return image

            center_x, center_y = image.shape[1] / 2, image.shape[0] / 2
            width_scaled, height_scaled = (
                image.shape[1] * self.__scale,
                image.shape[0] * self.__scale,
            )

            left_x, right_x = center_x - width_scaled / 2, center_x + width_scaled / 2
            top_y, bottom_y = center_y - height_scaled / 2, center_y + height_scaled / 2

            return image[int(top_y) : int(bottom_y), int(left_x) : int(right_x)]

        def _rotate_img(self, image):
            if self.__rotate_angle <= 0:
                return image
            height, width = image.shape[:2]
            center = (width / 2, height / 2)
            rotation_matrix = cv2.getRotationMatrix2D(center, self.__rotate_angle, 1.0)
            return cv2.warpAffine(image, rotation_matrix, (width, height))

        def _normalization_img(self, image):
            min_val = np.min(image)
            max_val = np.max(image)
            if max_val == 0:
                return np.zeros_like(image).astype(np.uint8)
            image = image - min_val
            image = image / max_val
            return (image * 255).astype(np.uint8)

        def _select_subset_image_files(self, image_files):
            if self.__enable_center_focus and self.__num_imgs is not None:
                middle = len(image_files) // 2
                num_imgs2 = self.__num_imgs // 2
                p1 = max(0, middle - num_imgs2)
                p2 = min(len(image_files), middle + num_imgs2)
                return image_files[p1:p2]
            elif self.__num_imgs is not None:
                return image_files[: self.__num_imgs]
            else:
                return image_files




In [4]:
class ScanDataset(Sequence):
    def __init__(
        self,
        dicom_loader,
        batch_size,
        subset="train",
        shuffle=True,
        debug_mode=False,
    ):
        self.__dicom_loader = dicom_loader
        self.__batch_size = batch_size
        self.__is_trainable = subset.lower() in ["validation", "train"]
        self.__shuffle = shuffle
        self.__debug = InternalDebug(debug_mode=debug_mode)
        self.__indices = np.arange(self.__dicom_loader.len)

        if self.__shuffle:
            np.random.shuffle(self.__indices)

    @property
    def workers(self):
        return 1

    @property
    def use_multiprocessing(self):
        return False

    @property
    def max_queue_size(self):
        return 8

    def on_epoch_end(self):
        if self.__shuffle:
            np.random.shuffle(self.__indices)

    def __getitem__(self, ids):
        from_id = ids * self.__batch_size
        to_id = (ids + 1) * self.__batch_size
        batch_indices = self.__indices[from_id:to_id]

        batches_y = []
        batches_x = [[] for _ in range(len(self.__dicom_loader.scan_categories))]

        for i in batch_indices:
            label = self.__dicom_loader.gel_label(i)
            batches_y.append(label)

            batch_x_image_paths = self.__dicom_loader.load_all_scans(
                i, show_progress=False
            )
            for j, (_, images) in enumerate(batch_x_image_paths.items()):
                batches_x[j].append(images)

        batch_x = tuple(np.array(b) for b in batches_x)
        batch_y = np.array(batches_y)

        if self.__is_trainable:
            return batch_x, batch_y
        else:
            return batch_x

    def __len__(self):
        return int(np.ceil(self.__dicom_loader.len / self.__batch_size))




In [5]:
class MRIType(Enum):
    FLAIR = "FLAIR"
    T1w = "T1w"
    T1wCE = "T1wCE"
    T2w = "T2w"


class DatasetType(Enum):
    TRAIN = "train"
    VALIDATION = "validation"
    TEST = "test"




In [6]:
VERSION = "V1"
VERBOSITY = 2
SEED = 123
SCAN_CATEGORIES = [mri_type.value for mri_type in MRIType]
EXCLUDED_IDS = [109, 123, 709]

RUN_DIR = "./run"
INPUT_PATH = "../input/rsna-miccai-brain-tumor-radiogenomic-classification"

TRAIN_DATASET_PATH = INPUT_PATH + "/train"
TRAIN_DATASET_DF_DIR = INPUT_PATH + "/train_labels.csv"

TEST_DATASET_PATH = INPUT_PATH + "/test"
TEST_DATASET_DF_DIR = INPUT_PATH + "/sample_submission.csv"

SUBMISSION_DATASET_DF_DIR = "/kaggle/working/submission.csv"

LOGS_PATH = f"{RUN_DIR}/logs"
BEST_MODEL_PATH = f"{RUN_DIR}/models"
BEST_MODEL_H5_DIR = f"{BEST_MODEL_PATH}/model_{VERSION}.h5"

MAX_THREADS_DICOM_LOADER = 8

IMG_WIDTH_SIZE, IMG_HEIGHT_SIZE, IMG_CHAN = (128, 128, 1)
IMG_SIZE = (IMG_WIDTH_SIZE, IMG_HEIGHT_SIZE)

IMG_SEQ = 32
IMG_SCALE = 0.90
IMG_ROTATE = 0
IMG_ENABLE_CENTRAL_FOCUS = True

SHUFFLE = True

INPUT_SHAPE = (IMG_WIDTH_SIZE, IMG_HEIGHT_SIZE, IMG_SEQ, IMG_CHAN)

MODEL_NAME = "Mult3DCNN4Input"
BATCH_SIZE = 8
EPOCHS = 26

COMPILE_OPTIMIZER = SGD(learning_rate=0.001)
COMPILE_LOSS = "binary_crossentropy"
COMPILE_METRICS = [AUC(name="auc")]

TF_CALL_BACK_BEST_MODEL_MONITOR = "val_auc"
TF_CALL_BACK_EARLY_STOP_PATIENTE = 6

os.makedirs(BEST_MODEL_PATH, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)

tf.config.optimizer.set_jit(False)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)



2026-05-13 03:58:49.461429: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [7]:
train_df = pd.read_csv(TRAIN_DATASET_DF_DIR)
train_df.rename(columns={"BraTS21ID": "ID", "MGMT_value": "Label"}, inplace=True)

index_to_remove = train_df[train_df["ID"].isin(EXCLUDED_IDS)].index
train_df.drop(index_to_remove, inplace=True)
train_df.reset_index(drop=True, inplace=True)

test_df = pd.read_csv(TEST_DATASET_DF_DIR)
test_df.rename(columns={"BraTS21ID": "ID", "MGMT_value": "Label"}, inplace=True)




In [8]:
class DeepScanModel(Model):
    def __init__(self, input_shape, model_name="My3DCNNModel"):
        self.input_layers = [Input(shape=input_shape) for _ in range(4)]
        self.cnn_models = [
            self.build_cnn_branch(input_layer) for input_layer in self.input_layers
        ]
        concatenated = concatenate(self.cnn_models)
        x = self.build_head(concatenated)
        super(DeepScanModel, self).__init__(
            inputs=self.input_layers, outputs=x, name=model_name
        )

    def build_cnn_branch(self, input_layer):
        x = Conv3D(64, 3)(input_layer)
        x = ReLU()(x)
        x = MaxPool3D(2)(x)
        x = BatchNormalization()(x)

        x = Conv3D(128, 3)(x)
        x = ReLU()(x)
        x = MaxPool3D(2)(x)
        x = BatchNormalization()(x)
        x = Dropout(0.1)(x)

        x = Conv3D(256, 3)(x)
        x = ReLU()(x)
        x = MaxPool3D(2)(x)
        x = BatchNormalization()(x)
        x = Dropout(0.2)(x)

        return x

    def build_head(self, x):
        x = GlobalAveragePooling3D()(x)
        x = Dense(1024)(x)
        x = ReLU()(x)
        x = Dropout(0.3)(x)
        x = Dense(1, activation="sigmoid")(x)
        return x




In [9]:
rng = np.random.RandomState(SEED)
y = train_df["Label"].values.astype(int)

idx0 = np.where(y == 0)[0]
idx1 = np.where(y == 1)[0]
rng.shuffle(idx0)
rng.shuffle(idx1)

split0 = int(0.85 * len(idx0))
split1 = int(0.85 * len(idx1))

train_idx = np.concatenate([idx0[:split0], idx1[:split1]])
val_idx = np.concatenate([idx0[split0:], idx1[split1:]])
rng.shuffle(train_idx)
rng.shuffle(val_idx)

train_split_df = train_df.iloc[train_idx].reset_index(drop=True)
val_split_df = train_df.iloc[val_idx].reset_index(drop=True)

CACHE_DIR = os.path.join(RUN_DIR, "dicom_cache_v1")
MEMORY_CACHE_SIZE = 250_000

train_dicom_loader = DICOMLoader(
    train_split_df,
    input_path=TRAIN_DATASET_PATH,
    scan_categories=SCAN_CATEGORIES,
    num_imgs=IMG_SEQ,
    size=IMG_SIZE,
    scale=IMG_SCALE,
    rotate_angle=IMG_ROTATE,
    max_threads=MAX_THREADS_DICOM_LOADER,
    enable_center_focus=IMG_ENABLE_CENTRAL_FOCUS,
    debug_mode=False,
    cache_dir=CACHE_DIR,
    memory_cache_size=MEMORY_CACHE_SIZE,
)

val_dicom_loader = DICOMLoader(
    val_split_df,
    input_path=TRAIN_DATASET_PATH,
    scan_categories=SCAN_CATEGORIES,
    num_imgs=IMG_SEQ,
    size=IMG_SIZE,
    scale=IMG_SCALE,
    rotate_angle=IMG_ROTATE,
    max_threads=MAX_THREADS_DICOM_LOADER,
    enable_center_focus=IMG_ENABLE_CENTRAL_FOCUS,
    debug_mode=False,
    cache_dir=CACHE_DIR,
    memory_cache_size=MEMORY_CACHE_SIZE,
)

train_dataset = ScanDataset(
    dicom_loader=train_dicom_loader,
    batch_size=BATCH_SIZE,
    subset=DatasetType.TRAIN.value,
    shuffle=True,
    debug_mode=False,
)

val_dataset = ScanDataset(
    dicom_loader=val_dicom_loader,
    batch_size=BATCH_SIZE,
    subset=DatasetType.VALIDATION.value,
    shuffle=False,
    debug_mode=False,
)

test_dicom_loader = DICOMLoader(
    test_df,
    input_path=TEST_DATASET_PATH,
    scan_categories=SCAN_CATEGORIES,
    num_imgs=IMG_SEQ,
    size=IMG_SIZE,
    scale=IMG_SCALE,
    rotate_angle=IMG_ROTATE,
    max_threads=MAX_THREADS_DICOM_LOADER,
    enable_center_focus=IMG_ENABLE_CENTRAL_FOCUS,
    debug_mode=False,
    cache_dir=CACHE_DIR,
    memory_cache_size=MEMORY_CACHE_SIZE,
)

test_dataset = ScanDataset(
    dicom_loader=test_dicom_loader,
    batch_size=BATCH_SIZE,
    subset=DatasetType.TEST.value,
    shuffle=False,
    debug_mode=False,
)

model = DeepScanModel(INPUT_SHAPE, model_name=MODEL_NAME)
model.compile(optimizer=COMPILE_OPTIMIZER, loss=COMPILE_LOSS, metrics=COMPILE_METRICS)

checkpoint_cb = ModelCheckpoint(
    filepath=BEST_MODEL_H5_DIR,
    monitor=TF_CALL_BACK_BEST_MODEL_MONITOR,
    mode="max",
    save_best_only=True,
    save_weights_only=False,
    verbose=1,
)

earlystop_cb = EarlyStopping(
    monitor=TF_CALL_BACK_BEST_MODEL_MONITOR,
    mode="max",
    patience=TF_CALL_BACK_EARLY_STOP_PATIENTE,
    restore_best_weights=True,
    verbose=1,
)

# Fix: Keras 3 TensorFlowTrainer.fit/predict do not accept workers/use_multiprocessing/max_queue_size.
# Removing these args is required for runtime; Sequence will still be consumed correctly (single-process).
history = None
try:
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        callbacks=[checkpoint_cb, earlystop_cb],
        verbose=VERBOSITY,
    )
except Exception as e:
    print("Training failed with exception; will proceed to submission fallback.")
    print(type(e).__name__, str(e))

if os.path.exists(BEST_MODEL_H5_DIR):
    model = load_model(
        BEST_MODEL_H5_DIR, custom_objects={"DeepScanModel": DeepScanModel}
    )




Training failed with exception; will proceed to submission fallback.
FileNotFoundError [Errno 2] No such file or directory: './run/dicom_cache_v1/4667146539792830983.npy.tmp' -> './run/dicom_cache_v1/4667146539792830983.npy'


In [10]:
def generate_predictions(model, test_dataset, test_df):
    # Fix: remove unsupported predict() kwargs under Keras 3.
    preds = model.predict(test_dataset, verbose=0)
    preds = np.asarray(preds).reshape(-1)

    submission = test_df.copy()
    submission["Label"] = preds[: len(submission)]
    submission.rename(columns={"ID": "BraTS21ID", "Label": "MGMT_value"}, inplace=True)
    return submission


def generate_fallback_submission(test_df, input_path):
    sample_sub_path = input_path + "/sample_submission.csv"
    sample_sub = pd.read_csv(sample_sub_path)
    out = sample_sub.copy()
    out["MGMT_value"] = 0.5
    return out


try:
    submission = generate_predictions(model, test_dataset, test_df)
except Exception as e:
    print("Prediction failed with exception; using 0.5 fallback submission.")
    print(type(e).__name__, str(e))
    submission = generate_fallback_submission(test_df, INPUT_PATH)

sample_sub_path = INPUT_PATH + "/sample_submission.csv"
sample_sub = pd.read_csv(sample_sub_path)
submission = sample_sub[["BraTS21ID"]].merge(
    submission[["BraTS21ID", "MGMT_value"]],
    on="BraTS21ID",
    how="left",
)

submission["MGMT_value"] = (
    submission["MGMT_value"].astype(float).fillna(0.5).clip(0.0, 1.0)
)



Prediction failed with exception; using 0.5 fallback submission.
FileNotFoundError [Errno 2] No such file or directory: './run/dicom_cache_v1/865895070736774491.npy.tmp' -> './run/dicom_cache_v1/865895070736774491.npy'


In [11]:
submission.info()
print(submission.head())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   BraTS21ID   59 non-null     int64  
 1   MGMT_value  59 non-null     float64
dtypes: float64(1), int64(1)
memory usage: 1.1 KB
   BraTS21ID  MGMT_value
0        356         0.5
1        140         0.5
2        757         0.5
3        275         0.5
4        570         0.5


In [12]:
submission.to_csv(SUBMISSION_DATASET_DF_DIR, index=False)
print(f"Wrote submission to: {SUBMISSION_DATASET_DF_DIR}")
print("File size (bytes):", os.path.getsize(SUBMISSION_DATASET_DF_DIR))
print("Submission columns:", submission.columns.tolist())
print("Num rows:", len(submission))

Wrote submission to: /kaggle/working/submission.csv
File size (bytes): 486
Submission columns: ['BraTS21ID', 'MGMT_value']
Num rows: 59
